# 5.3 — 図から根拠文へ


強い根拠文は、観察した値、比較範囲、対象期間、重要な限界を短く結びます。図から原因までは推測しません。


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

project = Path.cwd() / "projects" / "clinic-wait-evidence"
records = pd.read_csv(project / "data" / "clinic-waits-practice.csv")
service = records.groupby(["clinic_id", "clinic_name", "time_slot"], as_index=False).agg(
    patients_seen=("patients_seen", "sum"),
    total_wait_minutes=("total_wait_minutes", "sum"),
    over_60_minutes=("over_60_minutes", "sum"),
)
service["average_wait_minutes"] = service["total_wait_minutes"] / service["patients_seen"]
service["over_60_rate"] = service["over_60_minutes"] / service["patients_seen"] * 100
target = service.sort_values(["average_wait_minutes", "over_60_rate"], ascending=False).iloc[0]
display(target)


## 5.3.1 観察・解釈・因果を区別する


In [ ]:
observation = f"{target['clinic_name']} — {target['time_slot']} had an average wait of {target['average_wait_minutes']:.1f} minutes."
scope = f"The comparison covers {len(records)} clinic-time-week records from {records['week'].min()} to {records['week'].max()}."
limitation = "The records show where waiting was concentrated, but they do not identify the cause."
print(observation)
print(scope)
print(limitation)


## 5.3.2 判断に使う値を直接示す


In [ ]:
plot_data = service.assign(service=service["clinic_name"] + " — " + service["time_slot"]).sort_values("average_wait_minutes")
colours = ["#E45756" if value == target["clinic_name"] + " — " + target["time_slot"] else "#72B7B2" for value in plot_data["service"]]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(plot_data["service"], plot_data["average_wait_minutes"], color=colours)
ax.set(title="Average wait by clinic and time slot", xlabel="Average wait per patient (minutes)", ylabel="")
ax.set_xlim(left=0)
fig.tight_layout()
output = project / "output" / "lesson53_evidence.png"
output.parent.mkdir(exist_ok=True)
fig.savefig(output, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", output)


## 統合練習
上の図について、数値を含む観察、対象期間、原因を断定しない限界の三文を書いてください。保存したPNGを開き、表題・軸・単位が残っていることも確認します。
